# Extracción de Seguidores (TikTok, Twitter, Facebook)

Este notebook centraliza el scraping de seguidores de las tres redes sociales. Extrae la información desde el Excel original y guarda (o actualiza) los datos en un único archivo maestro: `resultados/seguidores.csv`.

In [5]:
import nest_asyncio
nest_asyncio.apply()

import time
import re
import random
import asyncio
import requests
from pathlib import Path
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

# ─── CONFIGURACIÓN GLOBAL ─────────────────────────────────────────
_SCRAPERS_DIR = Path().resolve()
_HERRAMIENTAS_DIR = _SCRAPERS_DIR.parent
_PROYECTO_DIR = _HERRAMIENTAS_DIR.parent
_RESULTADOS_DIR = _HERRAMIENTAS_DIR / "resultados"
_RESULTADOS_DIR.mkdir(exist_ok=True)

INPUT_FILE = _PROYECTO_DIR / "Colombia" / "Resultados electorales.xlsx"
SHEET_NAME = "Redes Sociales"
OUTPUT_CSV = _RESULTADOS_DIR / "seguidores.csv"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

def cargar_maestro(df_excel):
    if OUTPUT_CSV.exists():
        df_final = pd.read_csv(OUTPUT_CSV, sep=";")
    else:
        df_final = df_excel[["ID Candidato", "Candidato"]].copy()
    return df_final

def guardar_maestro(df_final):
    df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", sep=";")
    print(f"\n✅ Archivo maestro actualizado en: {OUTPUT_CSV}")

## 1. TikTok

In [6]:
def extract_tiktok_username(tiktok_url: str):
    if not tiktok_url or pd.isna(tiktok_url):
        return None
    match = re.search(r'tiktok\.com/@([\w.]+)', str(tiktok_url))
    if match: return match.group(1)
    return str(tiktok_url).strip('@').split('?')[0].strip('/')

async def scrape_tiktok_html(username: str, context):
    url = f"https://tokcount.com/?user={username}"
    page = await context.new_page()
    try:
        await page.goto(url, timeout=30000, wait_until="domcontentloaded")
        await asyncio.sleep(2)
        html = await page.content()
        match = re.search(r'"followers":(\d+)', html)
        if match:
            return int(match.group(1))
        return None
    except Exception as e:
        print(f"  Error con @{username}: {e}")
        return None
    finally:
        await page.close()

async def procesar_tiktok():
    print(f"Leyendo listado base...")
    df_excel = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, engine="openpyxl")
    df_excel["tiktok_username"] = df_excel["TikTok"].apply(extract_tiktok_username)
    candidatos = df_excel[df_excel["tiktok_username"].notna() & (df_excel["tiktok_username"] != "")]
    
    df_final = cargar_maestro(df_excel)
    for col in ["tiktok_username", "tiktok_followers", "tiktok_fecha_captura"]:
        if col not in df_final.columns: df_final[col] = None
        else: df_final[col] = df_final[col].astype(object)
            
    fecha = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    total = len(candidatos)
    print(f"Iniciando TikTok ({total} cuentas)...")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT, locale="es-CO")
        
        for i, (_, row) in enumerate(candidatos.iterrows(), 1):
            username = row['tiktok_username']
            id_cand = row['ID Candidato']
            print(f"[{i:3d}/{total}] @{username:<30} → ", end="", flush=True)
            
            followers = await scrape_tiktok_html(username, context)
            idx = df_final[df_final["ID Candidato"] == id_cand].index
            if idx.empty:
                df_final = pd.concat([df_final, pd.DataFrame([{"ID Candidato": id_cand, "Candidato": row["Candidato"]}])], ignore_index=True)
                idx = [len(df_final) - 1]
                
            df_final.loc[idx[0], "tiktok_username"] = username
            if followers is not None:
                df_final.loc[idx[0], "tiktok_followers"] = followers
                df_final.loc[idx[0], "tiktok_fecha_captura"] = fecha
                print(f"{followers:,} [OK]")
            else:
                print("No encontrado [X]")
                
            if i < total: await asyncio.sleep(random.uniform(3, 6))
        await browser.close()
        
    guardar_maestro(df_final)

# Descomenta la siguiente linea para ejecutar TikTok
await procesar_tiktok()

Leyendo listado base...
Iniciando TikTok (111 cuentas)...
[  1/111] @albertcorredor_                → 11,975 [OK]
[  2/111] @alcirasandovali                → 1,476 [OK]
[  3/111] @alejandrocharch                → 294,793 [OK]
[  4/111] @alejoeder                      → 116,508 [OK]
[  5/111] @alexander.baquero2             → 1,176 [OK]
[  6/111] @eribertoibarrac                → 385 [OK]
[  7/111] @betty.zorro                    → 449 [OK]
[  8/111] @antonio_bohorquez              → 8,748 [OK]
[  9/111] @camiloquirozh                  → 3,209 [OK]
[ 10/111] @carlosfgalan                   → 273,442 [OK]
[ 11/111] @carlosparrabga                 → 78,021 [OK]
[ 12/111] @carlosapinedocuello            → 2,967 [OK]
[ 13/111] @christianjmoreno               → 2,273 [OK]
[ 14/111] @claudiaandradegonzalez         → 1,076 [OK]
[ 15/111] @consueloberraca                → 1,165 [OK]
[ 16/111] @danis_renteria                 → 940 [OK]
[ 17/111] @dannycaiced0                   → 9,159 [OK]
[ 18/

## 2. Twitter

In [7]:
def extract_twitter_username(twitter_url: str):
    if not twitter_url or pd.isna(twitter_url): return None
    s = str(twitter_url).strip()
    if not s.startswith("http"): return s.lstrip("@").split("/")[0].strip() or None
    match = re.search(r'(?:twitter\.com|x\.com)/([A-Za-z0-9_]+)', s)
    return match.group(1) if match else None

def parse_tw_followers(text: str):
    if not text: return None
    clean = text.strip().replace(" ", "")
    if "," in clean and "." in clean:
        clean = clean.replace(",", "") if clean.rfind(".") > clean.rfind(",") else clean.replace(".", "")
        clean = clean.split(".")[0] if "." in clean else clean.split(",")[0]
    elif "." in clean:
        parts = clean.split(".")
        clean = clean.replace(".", "") if all(len(p) == 3 for p in parts[1:]) else parts[0]
    elif "," in clean: clean = clean.replace(",", "")
    try: return int(clean)
    except: return None

async def scrape_twitter_circleboom(page, username: str):
    try:
        await page.goto("https://circleboom.com/live-twitter-follower-counter", timeout=25000, wait_until="domcontentloaded")
        await asyncio.sleep(2)
        search = page.locator("#search")
        await search.wait_for(timeout=10000)
        await search.click()
        await search.evaluate("node => node.select()")
        await search.fill(username)
        await page.locator("button.content_search_btn__VRDNS").first.click()
        
        res_sel = "div.content_searched_user_id__fYj_P span"
        try: await page.wait_for_selector(res_sel, timeout=8000)
        except PWTimeout: await asyncio.sleep(8)
        
        raw = await page.locator(res_sel).first.inner_text()
        return parse_tw_followers(raw)
    except Exception as e:
        return None

async def procesar_twitter():
    print(f"Leyendo listado base...")
    df_excel = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, engine="openpyxl")
    df_excel["twitter_username"] = df_excel["X / Twitter"].apply(extract_twitter_username)
    candidatos = df_excel[df_excel["twitter_username"].notna() & (df_excel["twitter_username"] != "")]
    
    df_final = cargar_maestro(df_excel)
    for col in ["twitter_username", "twitter_followers", "twitter_fecha_captura"]:
        if col not in df_final.columns: df_final[col] = None
        else: df_final[col] = df_final[col].astype(object)

    fecha = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    total = len(candidatos)
    print(f"Iniciando Twitter ({total} cuentas)...")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT, locale="en-US")
        page = await context.new_page()
        
        for i, (_, row) in enumerate(candidatos.iterrows(), 1):
            username = row['twitter_username']
            id_cand = row['ID Candidato']
            print(f"[{i:3d}/{total}] @{username:<30} → ", end="", flush=True)
            
            followers = await scrape_twitter_circleboom(page, username)
            idx = df_final[df_final["ID Candidato"] == id_cand].index
            if idx.empty:
                df_final = pd.concat([df_final, pd.DataFrame([{"ID Candidato": id_cand, "Candidato": row["Candidato"]}])], ignore_index=True)
                idx = [len(df_final) - 1]
                
            df_final.loc[idx[0], "twitter_username"] = username
            if followers is not None:
                df_final.loc[idx[0], "twitter_followers"] = followers
                df_final.loc[idx[0], "twitter_fecha_captura"] = fecha
                print(f"{followers:,} [OK]")
            else:
                print("No encontrado [X]")
                
            if i < total: await asyncio.sleep(random.uniform(5, 10))
        await browser.close()
        
    guardar_maestro(df_final)

# Descomenta la siguiente linea para ejecutar Twitter
await procesar_twitter()

Leyendo listado base...
Iniciando Twitter (110 cuentas)...
[  1/110] @AdolfoRomeroB                  → 28 [OK]
[  2/110] @AlbertCorredor                 → 24,068 [OK]
[  3/110] @alcirasandovali                → 623 [OK]
[  4/110] @AlejandroChar                  → 499,679 [OK]
[  5/110] @alejoeder                      → 71,217 [OK]
[  6/110] @AlexBaqueroS                   → 5,416 [OK]
[  7/110] @betoibarra2020                 → 5 [OK]
[  8/110] @BettyZorro                     → 2,708 [OK]
[  9/110] @bohorquezesova                 → 2,037 [OK]
[ 10/110] @camiloquirozh                  → 4,270 [OK]
[ 11/110] @CarlosFGalan                   → 645,559 [OK]
[ 12/110] @CarlosParraBUC                 → 14,371 [OK]
[ 13/110] @CarlosPinedoC                  → 13,088 [OK]
[ 14/110] @CJoseMoreno                    → 29,196 [OK]
[ 15/110] @Claudia1Andrade                → 218 [OK]
[ 16/110] @consuelordonez                 → 6,077 [OK]
[ 17/110] @DanisRenteria                  → 4,029 [OK]
[ 18/110

## 3. Facebook

In [8]:
APIFY_TOKEN = "apify_api_EVQ13mkGT4eoE73NpplJhzZw6KWOLc1uParr"
ACTOR_ID    = "alpha-scraper~facebook-page-followers-following-count-scraper"

def extract_fb_username(fb_url: str):
    if pd.isna(fb_url): return None
    s = str(fb_url).strip()
    if not s.startswith("http") and "facebook.com" not in s: return s.strip("/")
    if "profile.php" in s: return None
    match = re.search(r'facebook\.com/([A-Za-z0-9._-]+)', s)
    if match:
        name = match.group(1)
        if name.lower() not in {"pages", "groups", "events", "watch", "login", "sharer", "profile.php"}:
            return name
    return None

def parse_fb_count(value):
    if value is None or str(value).strip() == "N/A": return None
    if isinstance(value, (int, float)): return int(value)
    s = str(value).strip().upper().replace(" ", "")
    for suffix, mult in {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}.items():
        if s.endswith(suffix):
            try: return int(float(s[:-1]) * mult)
            except: return None
    try: return int(s.replace(",", "").replace(".", ""))
    except: return None

def process_fb_batch(usernames: list):
    url_runs = f"https://api.apify.com/v2/acts/{ACTOR_ID}/runs"
    headers = {"Content-Type": "application/json"}
    params = {"token": APIFY_TOKEN}
    
    resp = requests.post(url_runs, json={"usernames": usernames}, headers=headers, params=params)
    if resp.status_code not in (200, 201): return {u: None for u in usernames}
    
    run_id = resp.json()["data"]["id"]
    elapsed = 0
    success = False
    while elapsed < 300:
        st = requests.get(f"https://api.apify.com/v2/actor-runs/{run_id}", params=params).json().get("data", {}).get("status", "")
        if st == "SUCCEEDED":
            success = True
            break
        if st in ("FAILED", "ABORTED", "TIMED-OUT"): break
        time.sleep(5)
        elapsed += 5
        
    results = {u: None for u in usernames}
    if success:
        items = requests.get(f"https://api.apify.com/v2/actor-runs/{run_id}/dataset/items", params={"token": APIFY_TOKEN, "format": "json"}).json()
        for item in items:
            uname = item.get("page_username") or item.get("username") or item.get("pageUsername")
            folls = item.get("followers_count") or item.get("followersCount") or item.get("followers")
            if uname and uname in results: results[uname] = parse_fb_count(folls)
    return results

def procesar_facebook():
    print(f"Leyendo listado base...")
    df_excel = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, engine="openpyxl")
    df_excel["fb_username"] = df_excel["Facebook"].apply(extract_fb_username)
    candidatos = df_excel[df_excel["fb_username"].notna() & (df_excel["fb_username"] != "")]
    unicos = candidatos["fb_username"].unique().tolist()
    
    df_final = cargar_maestro(df_excel)
    for col in ["facebook_username", "facebook_followers", "facebook_fecha_captura"]:
        if col not in df_final.columns: df_final[col] = None
        else: df_final[col] = df_final[col].astype(object)

    fecha = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    batch_size = 10
    batches = [unicos[i: i + batch_size] for i in range(0, len(unicos), batch_size)]
    print(f"Iniciando Facebook ({len(unicos)} cuentas en {len(batches)} lotes)...")

    todos_resultados = {}
    for i, b in enumerate(batches, 1):
        print(f"\nLote {i}/{len(batches)}...")
        res = process_fb_batch(b)
        todos_resultados.update(res)
        for u, f in res.items():
            print(f"   @{u:<30} → {f'{f:,} [OK]' if f is not None else 'No encontrado [X]'}")
        if i < len(batches): time.sleep(3)

    for _, row in candidatos.iterrows():
        username = row["fb_username"]
        id_cand = row["ID Candidato"]
        followers = todos_resultados.get(username)
        
        idx = df_final[df_final["ID Candidato"] == id_cand].index
        if idx.empty:
            df_final = pd.concat([df_final, pd.DataFrame([{"ID Candidato": id_cand, "Candidato": row["Candidato"]}])], ignore_index=True)
            idx = [len(df_final) - 1]
            
        df_final.loc[idx[0], "facebook_username"] = username
        if followers is not None:
            df_final.loc[idx[0], "facebook_followers"] = followers
            df_final.loc[idx[0], "facebook_fecha_captura"] = fecha

    guardar_maestro(df_final)

# Descomenta la siguiente linea para ejecutar Facebook
procesar_facebook()

Leyendo listado base...
Iniciando Facebook (121 cuentas en 13 lotes)...

Lote 1/13...
   @adolforomerob                  → 25,000 [OK]
   @albertyordanocorredor          → 18,000 [OK]
   @alcira.sandoval.56679          → 19,000 [OK]
   @AlejandroCharCh                → 465,000 [OK]
   @alejoederg                     → 168,000 [OK]
   @AlexBaqueroSa                  → 43,000 [OK]
   @eribertoibarrac                → 1,200 [OK]
   @BettyZorroA                    → 12,000 [OK]
   @antonio.e.collazos             → 7,000 [OK]
   @CamiloQuirozHinojosa           → 1,500 [OK]

Lote 2/13...
   @carlosfgalan                   → 199,000 [OK]
   @CarlosParraBUC                 → 216,000 [OK]
   @CarlosAlbertoPinedoCuello      → 13,000 [OK]
   @cristianjosemorenovillamizar   → 10,000 [OK]
   @Claudia.Andrade.2022           → 7,000 [OK]
   @ConsueloBerraca                → 1,500 [OK]
   @DanisRenteriaChala             → 14,000 [OK]
   @dannycaicedosoacha             → 19,000 [OK]
   @deninsonmendoza

ConnectionError: HTTPSConnectionPool(host='api.apify.com', port=443): Max retries exceeded with url: /v2/actor-runs/XPyZj3StTbXrvSaRr?token=apify_api_EVQ13mkGT4eoE73NpplJhzZw6KWOLc1uParr (Caused by NameResolutionError("HTTPSConnection(host='api.apify.com', port=443): Failed to resolve 'api.apify.com' ([Errno 11001] getaddrinfo failed)"))